# Byte Pair Encoder for Large Language Models

In [37]:
import re
from collections import defaultdict

In [38]:
def get_stats(ids,stats=None):

  if stats is not None:
    counts=stats
  else:
    counts=defaultdict(int)

  for pairs in zip(ids,ids[1:]): # sliding window of 2
    counts[pairs]+=1 # This shifts the list by 1 to make pairs and then adds them to count.
    # so count= {(a,b)=1} types. also as we imort default dict, it ensures u dont get error if pair not in counts

  return counts

In [39]:
def merge(ids,pair,idx):

  new_ids=[]
  i=0

  while(i<len(ids)-1):
    if (ids[i],ids[i+1])==pair:
      new_ids.append(idx)
      i+=2 # so if it is a pair, then it appends the pair, else it appends the individual index
      # standalone chars have already been appended to the list via self.vocab=self.build_base_vocab()
      #  So if ur most cpmmpn pair is lo, ur vocab has l: 108, o: 111, lo: 257
    else:
      new_ids.append(ids[i])
      i+=1
  if (i==len(ids)-1):
    new_ids.append(ids[i])

  return new_ids

In [40]:
class BPE_Encoder:
  def __init__(self):
    self.merges={}
    self.vocab=self.build_base_vocab() # base vocab for all 256 possible characters
    self.special_tokens={}

  def build_base_vocab(self):
    return {i:bytes([i]) for i in range(256)} # ensures all individual characters are already in list. Now it needs to add pairs

  def train(self,text,vocab_size,verbose=False):

    ids=list(text.encode("utf-8"))
    num_merges=vocab_size-256 # num merges since first 256 already taken

    for i in range(num_merges):

      stats=get_stats(ids) #makes the pairs

      if not stats:
        break

      best=max(stats,key=stats.get) # stats.get returns count for eah pair (value from key value pair).
      # This is for getting most commonly occuring token

      new_id=256+i

      self.merges[best]=new_id #encoding
      self.vocab[new_id]=self.vocab[best[0]]+self.vocab[best[1]] #best[0] is individual id eg 101. vocab[best[0]]='o' for 101

      ids=merge(ids,best,new_id) # to update the ids list with token pairs.

  def reg_special_tokens(self,special_tokens):
    # the special_tokens is a dictionary

    self.special_tokens=special_tokens

    for token,id in special_tokens.items():
      self.vocab[id]=token.encode("utf-8")

  def decode(self,ids):

    raw_bytes=b"".join(self.vocab[i] for i in ids) # b "".join cuz its a byte obj not a string
    return raw_bytes.decode("utf-8",errors="replace")

  def encode(self,text,allowed_special=None):

    if allowed_special is None:
      allowed_special=set()

    if allowed_special and self.special_tokens:
      # if both are non empty as in special tokens are being used and allowed
      active_special={
          k:v for k,v in self.special_tokens.items() #returns (key,val) for .items()
          if k in allowed_special # so if k is allowed and BEING actively used (if its in self.special or not)
          }
      if active_special:
        pattern="("+"|".join(re.escape(s) for s in active_special) + ")" # active_special is a dict so s is the string not its chars

        chunks=re.split(pattern,text)

      else:
        chunks=[text]
    else:
      chunks=[text]

    ids=[]

    for chunk in chunks:

      if chunk in self.special_tokens and chunk in allowed_special:

        ids.append(self.special_tokens[chunk]) # if its a special token, add the numeric value for it into ur ids

      else:

        chunk_ids=chunk.encode("utf-8") # gives binary string so b"hello"
        chunk_ids=list(chunk_ids) # gives list of values so [104,101,108,108,111]

        for pair,new_id in self.merges.items():
          # check against a key val pair of ((a,b),c) (a,b,c are all integers, theyre binary values for all chars)
          chunk_ids=merge(chunk_ids,pair,new_id) # edits the chunk list to reflect the pairs

        ids.extend(chunk_ids)

    return ids


In [41]:
# token="hello"
# token.encode("utf-8")

In [42]:
# new=(re.escape("|hello|"))
# new